# PatchInsight — treino gratuito no Kaggle / Colab

Notebook sem resultados pré-preenchidos. Usa o mesmo código do repositório.
Ative GPU e internet no Kaggle. No Colab use runtime gratuito com GPU, se disponível.
Envie e extraia o pacote `patchinsight-source-0.1.0.tar.gz` na área de trabalho.
Defina PROJECT abaixo para a pasta extraída contendo pyproject.toml.
Não envie o PDF privado. Nenhuma célula usa API paga.


In [ ]:
from pathlib import Path
import os, subprocess, sys
PROJECT = Path('/kaggle/working/patchinsight')  # Colab: /content/patchinsight
assert (PROJECT / 'pyproject.toml').exists(), 'Extraia o pacote de código e ajuste PROJECT.'
os.chdir(PROJECT)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
def run(*args):
    subprocess.run(list(args), check=True, cwd=PROJECT)


## Ambiente 3.11 e lock
A primeira execução precisa resolver uv.lock, que ainda está pendente no pacote inicial.
Baixe e versione o lock resolvido antes de considerar o experimento reproduzível.
As execuções seguintes usam o mesmo lock e --frozen.


In [ ]:
run(sys.executable, '-m', 'pip', 'install', 'uv==0.9.7')
run('uv', 'python', 'install', '3.11')
if not (PROJECT / 'uv.lock').exists():
    run('uv', 'lock', '--python', '3.11')
run('uv', 'sync', '--frozen', '--extra', 'ml', '--extra', 'dev')
run('uv', 'run', '--frozen', 'python', '-c',
    'import torch; print(torch.__version__); print(torch.cuda.is_available())')
DEVICE = 'cuda'  # Use cpu apenas se estiver disposto a esperar; sem garantia de tempo.


## Dados e testes
Falhar em integridade ou teste impede iniciar o experimento.
Os testes real_model só rodam após existir um checkpoint ajustado.


In [ ]:
run('uv', 'run', '--frozen', 'python', '-m', 'patchinsight.data')
run('uv', 'run', '--frozen', 'pytest')


## Treino e seleção exclusivamente por validação
Use RUN único para cada experimento. Não sobrescrever resultados para ocultar execuções.
O encoder e todos os pesos são ajustados no fine-tuning. A baseline mantém o encoder congelado.


In [ ]:
RUN = 'artifacts/run-42'
for method in ('heuristic', 'frozen', 'finetune'):
    run('uv', 'run', '--frozen', 'python', '-m', 'patchinsight.experiment', method,
        '--output', RUN, '--device', DEVICE)
os.environ['PATCHINSIGHT_MODEL'] = str(PROJECT / RUN / 'model')
run('uv', 'run', '--frozen', 'pytest', '-m', 'real_model')


## Avaliação final — só depois de encerrar as decisões
Leia training.json e as métricas de validação antes de ativar RUN_FINAL.
O teste já está congelado; não usar os erros para orientar novos dados ou ajustes.


In [ ]:
RUN_FINAL = False
if RUN_FINAL:
    run('uv', 'run', '--frozen', 'python', '-m', 'patchinsight.evaluate',
        '--run', RUN, '--device', DEVICE, '--final')


## Preservar artefatos fora da sessão
Faça download do zip e de uv.lock. Inclui checkpoints e metadados; não deixe a única cópia no runtime.
No Kaggle salve também uma versão do notebook com os outputs reais. No Colab baixe o zip ou copie
para um armazenamento próprio. Nenhuma publicação no Hugging Face é feita automaticamente aqui.


In [ ]:
import shutil
export_dir = PROJECT.parent / 'patchinsight-export'
export_dir.mkdir(exist_ok=True)
shutil.copytree(PROJECT / RUN, export_dir / 'run-42', dirs_exist_ok=True)
shutil.copy2(PROJECT / 'uv.lock', export_dir / 'uv.lock')
shutil.copy2(PROJECT / 'configs/experiment.json', export_dir / 'experiment.json')
shutil.copy2(PROJECT / 'data/freeze.json', export_dir / 'freeze.json')
if (PROJECT / 'reports/final').exists():
    shutil.copytree(PROJECT / 'reports/final', export_dir / 'final', dirs_exist_ok=True)
archive = shutil.make_archive(str(PROJECT.parent / 'patchinsight-artifacts'), 'zip', export_dir)
print('Baixe:', archive)
